# Demonstrate Various Data Preprocessing Techniques for a given Dataset

#### 1. Load the data

In [3]:
import pandas as pd

df = pd.read_csv('../ML Data.csv', dtype={'ROLL NO.': str})
print(df.head())
print(df.shape)

       ROLL NO. GENDER   SOFTWARE ENGINEER   TOC  COMPUTER NETWORK   \
0  240104010001       M               30.0  30.0               30.0   
1  240104010002       F               30.0  40.0               45.0   
2  240104010003       M               38.0  38.0               38.0   
3  240104010004       M               59.0  55.0               57.0   
4  240104010005       M               30.0  30.0               30.0   

   MACHINE LEARNING   DATA WAREHOUSE  GRADE  RESULT   
0               30.0             30.0      C    PASS  
1               40.0             35.0      B    Pass  
2               38.0             38.0      B    Pass  
3               58.0             54.0      A    pass  
4               30.0             30.0      C    Pass  
(86, 9)


**Reusing the same Dataset**

#### 2. Check for missing values

In [5]:
print(df.isnull().sum())

ROLL NO.             15
GENDER               14
SOFTWARE ENGINEER    18
TOC                  18
COMPUTER NETWORK     18
MACHINE LEARNING     18
DATA WAREHOUSE       18
GRADE                19
RESULT               18
dtype: int64


#### 3. Handle missing values

In [8]:
# Clean column names first (handles stray leading/trailing spaces in headers)
df.columns = df.columns.str.strip()

# Option A: Drop rows with any missing values
df_cleaned = df.dropna()

# Option B: Fill numeric columns with mean/median
df['MACHINE LEARNING'] = df['MACHINE LEARNING'].fillna(df['MACHINE LEARNING'].mean())

# Option C: Fill categorical columns with mode
df['GENDER'] = df['GENDER'].fillna(df['GENDER'].mode()[0])

#### 4. Remove duplicate records

In [9]:
# Check for duplicate rows
print("Number of duplicate rows:", df.duplicated().sum())

# Drop duplicate rows (keep the first occurrence)
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)

Number of duplicate rows: 13
Shape after removing duplicates: (73, 9)


#### 5. Standardize inconsistent categorical text
Notice `RESULT` has inconsistent casing (`PASS`, `Pass`, `pass`). Standardize text-based categorical columns.

In [10]:
# Standardize casing/whitespace in text categorical columns
df['RESULT'] = df['RESULT'].str.strip().str.upper()
df['GRADE'] = df['GRADE'].str.strip().str.upper()
df['GENDER'] = df['GENDER'].str.strip().str.upper()

print(df['RESULT'].value_counts())
print(df['GRADE'].value_counts())

RESULT
PASS         63
FAIL          4
DISCHARGE     1
Name: count, dtype: int64
GRADE
B     17
O     16
A     14
C      7
P      5
A+     4
F      2
1      1
O+     1
Name: count, dtype: int64


#### 6. Fill remaining missing values
Handle the remaining numeric and categorical columns that still have missing values.

In [11]:
numeric_cols = ['SOFTWARE ENGINEER', 'TOC', 'COMPUTER NETWORK', 'MACHINE LEARNING', 'DATA WAREHOUSE']

# Fill remaining numeric columns with their mean
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())

# Fill remaining categorical columns with mode
for col in ['GRADE', 'RESULT']:
    df[col] = df[col].fillna(df[col].mode()[0])

# ROLL NO. missing values: rows without an ID can't be reliably recovered, so drop them
df = df.dropna(subset=['ROLL NO.'])

print(df.isnull().sum())
print(df.shape)

ROLL NO.             0
GENDER               0
SOFTWARE ENGINEER    0
TOC                  0
COMPUTER NETWORK     0
MACHINE LEARNING     0
DATA WAREHOUSE       0
GRADE                0
RESULT               0
dtype: int64
(71, 9)


#### 7. Detect and handle outliers (IQR method)

In [12]:
import numpy as np

def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return data[(data[column] < lower_bound) | (data[column] > upper_bound)]

for col in numeric_cols:
    outliers = detect_outliers_iqr(df, col)
    print(f"{col}: {len(outliers)} outlier(s)")

SOFTWARE ENGINEER: 1 outlier(s)
TOC: 2 outlier(s)
COMPUTER NETWORK: 1 outlier(s)
MACHINE LEARNING: 3 outlier(s)
DATA WAREHOUSE: 2 outlier(s)


In [13]:
# Cap outliers to the IQR bounds instead of dropping them (retains sample size)
def cap_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    data[column] = data[column].clip(lower=lower_bound, upper=upper_bound)
    return data

for col in numeric_cols:
    df = cap_outliers_iqr(df, col)

print(df[numeric_cols].describe())

       SOFTWARE ENGINEER        TOC  COMPUTER NETWORK  MACHINE LEARNING  \
count          71.000000  71.000000         71.000000         71.000000   
mean           41.967067  43.236537         42.739022         43.900787   
std            14.329327  14.463834         15.962791         15.193373   
min            -3.750000  -1.250000         -9.000000          2.750000   
25%            31.500000  32.500000         30.000000         35.000000   
50%            41.852941  43.073529         43.000000         43.000000   
75%            55.000000  55.000000         56.000000         56.500000   
max            64.000000  88.750000         87.000000         88.750000   

       DATA WAREHOUSE  
count       71.000000  
mean        43.747307  
std         14.202961  
min          5.000000  
25%         35.000000  
50%         43.764706  
75%         55.000000  
max         85.000000  


#### 8. Encode categorical variables

In [17]:
# Label Encoding for binary/ordinal-like categories
from sklearn.preprocessing import LabelEncoder

le_gender = LabelEncoder()
df['GENDER_ENCODED'] = le_gender.fit_transform(df['GENDER'])

le_result = LabelEncoder()
df['RESULT_ENCODED'] = le_result.fit_transform(df['RESULT'])

# Ordinal encoding for GRADE since grades have a natural order
grade_order = {'F': 0, 'D': 1, 'C': 2, 'B': 3, 'A': 4, 'O': 5}
df['GRADE_ENCODED'] = df['GRADE'].map(grade_order)

print(df[['GENDER', 'GENDER_ENCODED', 'RESULT', 'RESULT_ENCODED', 'GRADE', 'GRADE_ENCODED']].head())

  GENDER  GENDER_ENCODED RESULT  RESULT_ENCODED GRADE  GRADE_ENCODED
0      M               2   PASS               2     C            2.0
1      F               0   PASS               2     B            3.0
2      M               2   PASS               2     B            3.0
3      M               2   PASS               2     A            4.0
4      M               2   PASS               2     C            2.0


#### 9. Feature scaling (Normalization & Standardization)

In [16]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Min-Max Normalization -> scales values to [0, 1]
minmax_scaler = MinMaxScaler()
df_normalized = df.copy()
df_normalized[numeric_cols] = minmax_scaler.fit_transform(df[numeric_cols])

# Standardization -> mean 0, std 1 (Z-score scaling)
standard_scaler = StandardScaler()
df_standardized = df.copy()
df_standardized[numeric_cols] = standard_scaler.fit_transform(df[numeric_cols])

print("Normalized sample:\n", df_normalized[numeric_cols].head())
print("\nStandardized sample:\n", df_standardized[numeric_cols].head())

Normalized sample:
    SOFTWARE ENGINEER       TOC  COMPUTER NETWORK  MACHINE LEARNING  \
0           0.498155  0.347222          0.406250          0.316860   
1           0.498155  0.458333          0.562500          0.433140   
2           0.616236  0.436111          0.489583          0.409884   
3           0.926199  0.625000          0.687500          0.642442   
4           0.498155  0.347222          0.406250          0.316860   

   DATA WAREHOUSE  
0          0.3125  
1          0.3750  
2          0.4125  
3          0.6125  
4          0.3125  

Standardized sample:
    SOFTWARE ENGINEER       TOC  COMPUTER NETWORK  MACHINE LEARNING  \
0          -0.841089 -0.921661         -0.803725         -0.921436   
1          -0.841089 -0.225360          0.142649         -0.258570   
2          -0.278820 -0.364620         -0.298992         -0.391143   
3           1.197137  0.819091          0.899747          0.934589   
4          -0.841089 -0.921661         -0.803725         -0.921436

#### 10. Final preprocessed dataset

In [18]:
print("Final cleaned dataset shape:", df.shape)
print(df.dtypes)
df.head()

Final cleaned dataset shape: (71, 12)
ROLL NO.                 str
GENDER                   str
SOFTWARE ENGINEER    float64
TOC                  float64
COMPUTER NETWORK     float64
MACHINE LEARNING     float64
DATA WAREHOUSE       float64
GRADE                    str
RESULT                   str
GENDER_ENCODED         int64
RESULT_ENCODED         int64
GRADE_ENCODED        float64
dtype: object


,ROLL NO.,GENDER,SOFTWARE ENGINEER,TOC,COMPUTER NETWORK,MACHINE LEARNING,DATA WAREHOUSE,GRADE,RESULT,GENDER_ENCODED,RESULT_ENCODED,GRADE_ENCODED
0,240104010001,M,30.0,30.0,30.0,30.0,30.0,C,PASS,2,2,2.0
1,240104010002,F,30.0,40.0,45.0,40.0,35.0,B,PASS,0,2,3.0
2,240104010003,M,38.0,38.0,38.0,38.0,38.0,B,PASS,2,2,3.0
3,240104010004,M,59.0,55.0,57.0,58.0,54.0,A,PASS,2,2,4.0
4,240104010005,M,30.0,30.0,30.0,30.0,30.0,C,PASS,2,2,2.0
